In [1]:
import sys
import os
import chromadb

# Добавляем корень проекта в sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from scripts.pplx_embed import PplxEmbedFunction, PplxContextEmbedFunction

/Users/sergey/Desktop/Moodle_RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
ORIGINALS_DIR = "/Users/sergey/Desktop/Moodle_RAG/data/moodle_docs/clean_markdown"
CHUNKS_DIR = "/Users/sergey/Desktop/Moodle_RAG/data/moodle_docs/chunks_md"
CHROMA_PATH = "/Users/sergey/Desktop/Moodle_RAG/scripts/Notebooks/chroma_db_pplx"
COLLECTION_NAME = "moodle_docs_pplx"

In [12]:
query_embed = PplxEmbedFunction()
ctx_embed = PplxContextEmbedFunction()

print(f"Query model device: {query_embed.device}")
print(f"Context model device: {ctx_embed.device}")

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 1953.15it/s, Materializing param=norm.weight]                              
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 310/310 [00:00<00:00, 1520.12it/s, Materializing param=norm.weight]                              


Query model device: mps
Context model device: cpu


In [14]:
client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=query_embed
)

In [ ]:
total_chunks = 0

# Перебираем папки с чанками
for doc_folder in sorted(os.listdir(CHUNKS_DIR)):
    doc_folder_path = os.path.join(CHUNKS_DIR, doc_folder)
    if not os.path.isdir(doc_folder_path):
        continue

    # Находим оригинальный .md файл
    original_path = os.path.join(ORIGINALS_DIR, doc_folder + ".md")
    if os.path.exists(original_path):
        full_document = open(original_path, "r", encoding="utf-8").read()
    else:
        print(f"⚠️  Оригинал не найден: {original_path}, пропускаю")
        continue

    # Собираем все чанки из папки
    chunk_files = sorted([
        f for f in os.listdir(doc_folder_path) if f.endswith(".md")
    ])

    if not chunk_files:
        continue

    chunks = []
    chunk_ids = []
    metadatas = []

    for chunk_file in chunk_files:
        chunk_path = os.path.join(doc_folder_path, chunk_file)
        chunk_text = open(chunk_path, "r", encoding="utf-8").read()

        chunks.append(chunk_text)
        chunk_ids.append(f"{doc_folder}::{chunk_file}")
        metadatas.append({
            "source": doc_folder + ".md",
            "chunk_file": chunk_file,
            "chunk_index": int(chunk_file.replace("chunk_", "").replace(".md", ""))
        })

    # Эмбеддинги с контекстом полного документа
    embeddings = ctx_embed.embed_with_context(chunks, full_document)

    collection.add(
        documents=chunks,
        embeddings=embeddings,
        ids=chunk_ids,
        metadatas=metadatas
    )

    total_chunks += len(chunks)
    print(f"✅ {doc_folder}: {len(chunks)} чанков")

print(f"\n📦 Всего проиндексировано: {total_chunks} чанков")
print(f"📦 В коллекции: {collection.count()} чанков")


In [ ]:
print(f"📦 Чанков в коллекции: {collection.count()}")
print("=" * 60)

# Тестовые запросы
queries = [
    "Как создать новый курс в Moodle?",
    "How to set up gradebook?",
    "Как добавить студента на курс?",
]

for query in queries:
    print(f"\n🔍 Запрос: {query}")
    print("-" * 60)

    results = collection.query(
        query_texts=[query],
        n_results=5
    )

    for i, (doc, meta, dist) in enumerate(zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    )):
        print(f"\n  #{i+1} | Расстояние: {dist:.4f} | Источник: {meta['source']}")
        print(f"  Чанк: {meta['chunk_file']}")
        print(f"  Текст: {doc[:200]}...")

    print("=" * 60)

📦 Чанков в коллекции: 3395

🔍 Запрос: Как создать новый курс в Moodle?
------------------------------------------------------------


/Users/sergey/.cache/huggingface/modules/transformers_modules/perplexity_hyphen_ai/pplx_hyphen_embed_hyphen_v1_hyphen_0_dot_6b/1dc2ea99a948a2f17b103949ad02b0194a20c0a8/modeling.py:62: FutureWarning: `input_embeds` is deprecated and will be removed in version 5.6.0 for `create_causal_mask`. Use `inputs_embeds` instead.
  "full_attention": create_causal_mask(



  #1 | Расстояние: 0.7748 | Источник: 403__en__Adding_a_new_course.md
  Чанк: chunk_0000.md
  Текст: # Adding a new course

## Adding a course
By default a regular teacher can't add a new course. To add a new course to Moodle, you need to have either Administrator, Course Creator or Manager rights.To...

  #2 | Расстояние: 0.8110 | Источник: 403__en__Adding_a_new_course.md
  Чанк: chunk_0010.md
  Текст: ## Media
### YouTube
- https://youtu.be/MzK2jb-9SwE
### Images
- https://docs.moodle.org/403/en/images_en/1/14/26defaultcoursevalues.png
- https://docs.moodle.org/403/en/images_en/6/60/coursesort.png
...

  #3 | Расстояние: 0.8143 | Источник: 403__en__Adding_a_new_course.md
  Чанк: chunk_0011.md
  Текст: - https://docs.moodle.org/403/en/images_en/thumb/7/7c/26addcourse1.png/400px-26addcourse1.png
- https://docs.moodle.org/403/en/images_en/thumb/7/7c/26addcourse1.png/600px-26addcourse1.png
- https://do...

  #4 | Расстояние: 0.8262 | Источник: 403__en__Adding_a_new_course.md
  Чанк: chu